# 01. Importing Libraries

In [1]:
import os
import math
import random
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import scipy
import numpy as np
import pandas as pd

# 02. Importing Data

In [2]:
data = pd.read_csv("../Data/Clean_data/final_test_data.csv")

In [3]:
data.shape

(317235, 7)

In [35]:
data.head()

,client_id,visitor_id,visit_id,process_step,date_time,source,group_test
0,3561384,451664975_1722933822,100012776_37918976071_457913,confirm,2017-04-26 13:22:17,pt1,test
1,3561384,451664975_1722933822,100012776_37918976071_457913,confirm,2017-04-26 13:23:09,pt1,test
2,7338123,612065484_94198474375,100019538_17884295066_43909,start,2017-04-09 16:20:56,pt1,test
3,7338123,612065484_94198474375,100019538_17884295066_43909,step_1,2017-04-09 16:21:12,pt1,test
4,7338123,612065484_94198474375,100019538_17884295066_43909,step_2,2017-04-09 16:21:21,pt1,test


In [4]:
data['process_step'].value_counts()

process_step
start      101153
step_1      68210
step_2      56672
step_3      48264
confirm     42936
Name: count, dtype: int64

In [5]:
data.dtypes

client_id       int64
visitor_id        str
visit_id          str
process_step      str
date_time         str
source            str
group_test        str
dtype: object

In [6]:
# converting date_time to datetime format
data['date_time'] = pd.to_datetime(data['date_time'], format='%Y-%m-%d %H:%M:%S')

In [8]:
data.dtypes

client_id                int64
visitor_id                 str
visit_id                   str
process_step               str
date_time       datetime64[us]
source                     str
group_test                 str
dtype: object

In [10]:
data.head()

,client_id,visitor_id,visit_id,process_step,date_time,source,group_test
0,3561384,451664975_1722933822,100012776_37918976071_457913,confirm,2017-04-26 13:22:17,pt1,test
1,3561384,451664975_1722933822,100012776_37918976071_457913,confirm,2017-04-26 13:23:09,pt1,test
2,7338123,612065484_94198474375,100019538_17884295066_43909,start,2017-04-09 16:20:56,pt1,test
3,7338123,612065484_94198474375,100019538_17884295066_43909,step_1,2017-04-09 16:21:12,pt1,test
4,7338123,612065484_94198474375,100019538_17884295066_43909,step_2,2017-04-09 16:21:21,pt1,test


In [11]:
# Mapping process steps to numeric values for easier analysis
step_order = {'start': 0, 'step_1': 1, 'step_2': 2, 'step_3': 3, 'confirm': 4}

data['step_num'] = data['process_step'].map(step_order)

In [12]:
# checking the new column
data

,client_id,visitor_id,visit_id,process_step,date_time,source,group_test,step_num
0,3561384,451664975_1722933822,100012776_37918976071_457913,confirm,2017-04-26 13:22:17,pt1,test,4
1,3561384,451664975_1722933822,100012776_37918976071_457913,confirm,2017-04-26 13:23:09,pt1,test,4
2,7338123,612065484_94198474375,100019538_17884295066_43909,start,2017-04-09 16:20:56,pt1,test,0
3,7338123,612065484_94198474375,100019538_17884295066_43909,step_1,2017-04-09 16:21:12,pt1,test,1
4,7338123,612065484_94198474375,100019538_17884295066_43909,step_2,2017-04-09 16:21:21,pt1,test,2
...,...,...,...,...,...,...,...,...
317230,6627522,730634087_44272418812,999988789_76411676596_272843,start,2017-04-21 23:49:11,pt1,test,0
317231,6627522,730634087_44272418812,999988789_76411676596_272843,step_1,2017-04-21 23:49:22,pt1,test,1
317232,6627522,730634087_44272418812,999988789_76411676596_272843,step_2,2017-04-21 23:50:16,pt1,test,2
317233,6627522,730634087_44272418812,999988789_76411676596_272843,step_1,2017-04-21 23:51:00,pt1,test,1


### KPI 2: time spent on each step

In [13]:
# creating new column of the time of the next step for each visit
data['time_next'] = data.groupby('visit_id')['date_time'].shift(-1)

# calculating time spent on each step in seconds
data['time_spent'] = (data['time_next'] - data['date_time']).dt.total_seconds()

# remonving rows with time spent greater than 30 minutes (1800 seconds) as they are likely outliers
data_time = data[data['time_spent'] < 1800]

In [14]:
# checking creation of new columns
data

,client_id,visitor_id,visit_id,process_step,date_time,source,group_test,step_num,time_next,time_spent
0,3561384,451664975_1722933822,100012776_37918976071_457913,confirm,2017-04-26 13:22:17,pt1,test,4,2017-04-26 13:23:09,52.0
1,3561384,451664975_1722933822,100012776_37918976071_457913,confirm,2017-04-26 13:23:09,pt1,test,4,NaT,NaN
2,7338123,612065484_94198474375,100019538_17884295066_43909,start,2017-04-09 16:20:56,pt1,test,0,2017-04-09 16:21:12,16.0
3,7338123,612065484_94198474375,100019538_17884295066_43909,step_1,2017-04-09 16:21:12,pt1,test,1,2017-04-09 16:21:21,9.0
4,7338123,612065484_94198474375,100019538_17884295066_43909,step_2,2017-04-09 16:21:21,pt1,test,2,2017-04-09 16:21:35,14.0
...,...,...,...,...,...,...,...,...,...,...
317230,6627522,730634087_44272418812,999988789_76411676596_272843,start,2017-04-21 23:49:11,pt1,test,0,2017-04-21 23:49:22,11.0
317231,6627522,730634087_44272418812,999988789_76411676596_272843,step_1,2017-04-21 23:49:22,pt1,test,1,2017-04-21 23:50:16,54.0
317232,6627522,730634087_44272418812,999988789_76411676596_272843,step_2,2017-04-21 23:50:16,pt1,test,2,2017-04-21 23:51:00,44.0
317233,6627522,730634087_44272418812,999988789_76411676596_272843,step_1,2017-04-21 23:51:00,pt1,test,1,2017-04-21 23:51:09,9.0


In [17]:
# creation of a new dataset focused on group, steps and avg time spent on each step
time_kpi = data_time.groupby(['group_test', 'process_step'])['time_spent'].mean().round(2).reset_index()
time_kpi.columns = ['group', 'process_step', 'avg_time_seconds']

# creating new column for average time in minutes for easier interpretation
time_kpi['avg_time_minutes'] = (time_kpi['avg_time_seconds'] / 60).round(2)

# creating new column for step number to sort the steps in the correct order per group
time_kpi['step_num'] = time_kpi['process_step'].map(step_order)
time_kpi = time_kpi.sort_values(['step_num', 'group'])

In [18]:
# checking new dataframe for KPI 2
time_kpi

,group,process_step,avg_time_seconds,avg_time_minutes,step_num
1,control,start,60.91,1.02,0
6,test,start,56.24,0.94,0
2,control,step_1,47.46,0.79,1
7,test,step_1,58.48,0.97,1
3,control,step_2,90.21,1.50,2
8,test,step_2,88.13,1.47,2
4,control,step_3,133.30,2.22,3
9,test,step_3,124.83,2.08,3
0,control,confirm,156.16,2.60,4
5,test,confirm,221.59,3.69,4


### KPI 3: error rate (users going back to a previous step)

In [19]:
# creating new column to identify if there are any errors in the process (going back to a previous step)
data['prev_step_num'] = data.groupby('visit_id')['step_num'].shift(1)

# creating label for error if the current step number is less than the previous step number (indicating a step back)
data['is_error'] = data['step_num'] < data['prev_step_num']


In [48]:
# checking creation of new columns
data.head()

,client_id,visitor_id,visit_id,process_step,date_time,source,group_test,step_num,time_next,time_spent,prev_step_num,is_error
0,3561384,451664975_1722933822,100012776_37918976071_457913,confirm,2017-04-26 13:22:17,pt1,test,4,2017-04-26 13:23:09,52.0,NaN,False
1,3561384,451664975_1722933822,100012776_37918976071_457913,confirm,2017-04-26 13:23:09,pt1,test,4,NaT,NaN,4.0,False
2,7338123,612065484_94198474375,100019538_17884295066_43909,start,2017-04-09 16:20:56,pt1,test,0,2017-04-09 16:21:12,16.0,NaN,False
3,7338123,612065484_94198474375,100019538_17884295066_43909,step_1,2017-04-09 16:21:12,pt1,test,1,2017-04-09 16:21:21,9.0,0.0,False
4,7338123,612065484_94198474375,100019538_17884295066_43909,step_2,2017-04-09 16:21:21,pt1,test,2,2017-04-09 16:21:35,14.0,1.0,False


In [20]:
# creating new dataset focused on group, steps and error rate
error_kpi = data.groupby(['group_test', 'process_step']).agg(
    total_visits=('visit_id', 'count'),
    errors=('is_error', 'sum')
).reset_index()

# calculating error rate as percentage of errors out of total visits for each group and step
error_kpi['error_rate'] = (error_kpi['errors'] / error_kpi['total_visits'] * 100).round(2)

# creating new column for step number to sort the steps in the correct order per group
error_kpi['step_num'] = error_kpi['process_step'].map(step_order)
error_kpi = error_kpi.sort_values(['step_num', 'group_test'])

In [21]:
# checking new dataframe for KPI 3
# dataframe shows that, 10% of the times start was visited, users came there back from another step
error_kpi

,group_test,process_step,total_visits,errors,error_rate,step_num
1,control,start,45380,4932,10.87,0
6,test,start,55773,10621,19.04,0
2,control,step_1,29544,2303,7.80,1
7,test,step_1,38666,3414,8.83,1
3,control,step_2,25773,2364,9.17,2
8,test,step_2,30899,2283,7.39,2
4,control,step_3,22503,101,0.45,3
9,test,step_3,25761,22,0.09,3
0,control,confirm,17336,0,0.00,4
5,test,confirm,25600,0,0.00,4


In [22]:
# checking the error based on the step that people left from
step_order_reversed = {0: 'start', 1: 'step_1', 2: 'step_2', 3: 'step_3', 4: 'confirm'}
data['prev_process_step'] = data['prev_step_num'].map(step_order_reversed)

In [23]:
data.head()

,client_id,visitor_id,visit_id,process_step,date_time,source,group_test,step_num,time_next,time_spent,prev_step_num,is_error,prev_process_step
0,3561384,451664975_1722933822,100012776_37918976071_457913,confirm,2017-04-26 13:22:17,pt1,test,4,2017-04-26 13:23:09,52.0,NaN,False,NaN
1,3561384,451664975_1722933822,100012776_37918976071_457913,confirm,2017-04-26 13:23:09,pt1,test,4,NaT,NaN,4.0,False,confirm
2,7338123,612065484_94198474375,100019538_17884295066_43909,start,2017-04-09 16:20:56,pt1,test,0,2017-04-09 16:21:12,16.0,NaN,False,NaN
3,7338123,612065484_94198474375,100019538_17884295066_43909,step_1,2017-04-09 16:21:12,pt1,test,1,2017-04-09 16:21:21,9.0,0.0,False,start
4,7338123,612065484_94198474375,100019538_17884295066_43909,step_2,2017-04-09 16:21:21,pt1,test,2,2017-04-09 16:21:35,14.0,1.0,False,step_1


In [24]:
# creating new dataset grouping by the step users LEFT from (not where they landed)
error_kpi_from = data.groupby(['group_test', 'prev_process_step']).agg(
    total_visits=('visit_id', 'count'),
    errors=('is_error', 'sum')
).reset_index()

# calculating error rate as percentage
error_kpi_from['error_rate_from'] = (error_kpi_from['errors'] / error_kpi_from['total_visits'] * 100).round(2)

# creating new column for step number to sort the steps in the correct order per group
error_kpi_from['step_num'] = error_kpi_from['prev_process_step'].map(step_order)
error_kpi_from = error_kpi_from.sort_values(['step_num', 'group_test'])

In [ ]:
# checking new dataframe for KPI 3
# dataframe shows that 39.03 of control on confirm went back to steps before
error_kpi_from

,group_test,prev_process_step,total_visits,errors,error_rate_from,step_num
1,control,start,35741,0,0.00,0
6,test,start,46326,0,0.00,0
2,control,step_1,26046,2493,9.57,1
7,test,step_1,35535,6410,18.04,1
3,control,step_2,24316,2167,8.91,2
8,test,step_2,29576,4783,16.17,2
4,control,step_3,20281,4247,20.94,3
9,test,step_3,23984,4744,19.78,3
0,control,confirm,2032,793,39.03,4
5,test,confirm,4193,403,9.61,4


# 03. Exporting data pieces

In [27]:
time_kpi.to_csv('../Data/Data_pieces/time_kpi.csv', index=False)

In [28]:
error_kpi_from.to_csv('../Data/Data_pieces/error_kpi_from.csv', index=False)

In [29]:
error_kpi.to_csv('../Data/Data_pieces/error_kpi.csv', index=False)

In [31]:
data.to_csv('../Data/Clean_data/final_test_data_wrangled.csv', index=False)

# 04. Checking how time_spent shows if only 1 row per visit

In [37]:
data[data['process_step'] == 'confirm'].groupby('visit_id')['process_step'].count()

visit_id
100012776_37918976071_457913    2
100019538_17884295066_43909     1
100022086_87870757897_149620    1
10006594_66157970412_679648     1
10007589_47780784567_391490     1
                               ..
999958344_67534252886_39917     2
999971096_28827267783_236076    1
999976049_95772503197_182554    1
999984454_18731538378_781808    1
999985675_64610694964_443659    1
Name: process_step, Length: 37680, dtype: int64

In [ ]:
# checking visits with only 1 confirm step
confirm_counts = data[data['process_step'] == 'confirm'].groupby('visit_id')['process_step'].count()
single_confirm_visits = confirm_counts[confirm_counts == 1].index
resultdata = data[(data['visit_id'].isin(single_confirm_visits)) & (data['process_step'] == 'confirm')]
resultdata

,client_id,visitor_id,visit_id,process_step,date_time,source,group_test,step_num,time_next,time_spent,prev_step_num,is_error,prev_process_step
12,7338123,612065484_94198474375,100019538_17884295066_43909,confirm,2017-04-09 16:24:58,pt1,test,4,NaT,NaN,3.0,False,step_3
17,2478628,754122351_18568832435,100022086_87870757897_149620,confirm,2017-05-23 20:47:01,pt2,test,4,NaT,NaN,3.0,False,step_3
35,3479519,194422203_56127484794,10006594_66157970412_679648,confirm,2017-04-13 11:56:12,pt1,control,4,2017-04-13 11:56:12,0.0,3.0,False,step_3
42,5477656,164180384_39691984082,10007589_47780784567_391490,confirm,2017-05-18 08:03:33,pt2,control,4,NaT,NaN,3.0,False,step_3
51,8631696,429350107_31978453627,100173292_91322748906_143563,confirm,2017-04-25 10:30:30,pt1,test,4,NaT,NaN,3.0,False,step_3
...,...,...,...,...,...,...,...,...,...,...,...,...,...
317179,5353871,935108910_78972268403,999891710_95999857132_598498,confirm,2017-04-12 15:16:21,pt1,test,4,NaT,NaN,3.0,False,step_3
317213,2979920,830229399_73416253406,999971096_28827267783_236076,confirm,2017-04-13 10:34:08,pt1,test,4,NaT,NaN,3.0,False,step_3
317219,4449968,842902495_57580498240,999976049_95772503197_182554,confirm,2017-04-04 13:02:18,pt1,test,4,NaT,NaN,3.0,False,step_3
317224,829911,648229874_89449279372,999984454_18731538378_781808,confirm,2017-03-29 11:21:07,pt1,test,4,NaT,NaN,3.0,False,step_3


In [ ]:
# only 600 visits with only 1 confirm step have time spent data, which is a small part of the total sample size
resultdata['time_spent'].isnull().sum()

np.int64(33037)

In [ ]:
# checking visits with more than 1 confirm step to confirm if they have time spent data on first confirm step
confirm_counts = data[data['process_step'] == 'confirm'].groupby('visit_id')['process_step'].count()
multiple_confirm_visits = confirm_counts[confirm_counts > 1].index
data[(data['visit_id'].isin(multiple_confirm_visits)) & (data['process_step'] == 'confirm')]

,client_id,visitor_id,visit_id,process_step,date_time,source,group_test,step_num,time_next,time_spent,prev_step_num,is_error,prev_process_step
0,3561384,451664975_1722933822,100012776_37918976071_457913,confirm,2017-04-26 13:22:17,pt1,test,4,2017-04-26 13:23:09,52.0,NaN,False,NaN
1,3561384,451664975_1722933822,100012776_37918976071_457913,confirm,2017-04-26 13:23:09,pt1,test,4,NaT,NaN,4.0,False,confirm
89,1562128,297078435_42288608841,100258507_71262593004_214494,confirm,2017-03-29 12:48:24,pt1,test,4,2017-03-29 12:49:16,52.0,3.0,False,step_3
90,1562128,297078435_42288608841,100258507_71262593004_214494,confirm,2017-03-29 12:49:16,pt1,test,4,2017-03-29 12:49:32,16.0,4.0,False,confirm
91,1562128,297078435_42288608841,100258507_71262593004_214494,confirm,2017-03-29 12:49:32,pt1,test,4,NaT,NaN,4.0,False,confirm
...,...,...,...,...,...,...,...,...,...,...,...,...,...
317156,426074,971489883_58289393614,999841812_92641758733_294220,confirm,2017-04-05 11:54:37,pt1,test,4,NaT,NaN,4.0,False,confirm
317193,9498187,599653496_46358190244,999954858_74676709104_879685,confirm,2017-04-05 11:14:59,pt1,test,4,2017-04-05 11:15:41,42.0,3.0,False,step_3
317194,9498187,599653496_46358190244,999954858_74676709104_879685,confirm,2017-04-05 11:15:41,pt1,test,4,NaT,NaN,4.0,False,confirm
317199,8971313,520929316_99288864740,999958344_67534252886_39917,confirm,2017-04-15 00:35:54,pt1,test,4,2017-04-15 00:36:59,65.0,3.0,False,step_3
